In [1]:

import pandas as pd
import numpy as np
import os

df = pd.read_csv(
    "../data/interim/maternal_health_initial_clean.csv"
)

# Create Pulse Pressure
df["Pulse_Pressure"] = (
    df["Systolic Blood Pressure(mm Hg)"]
    - df["Diastolic Blood Pressure(mm Hg)"]
)

# Create Mean Arterial Pressure
df["Mean_Arterial_Pressure"] = (
    df["Systolic Blood Pressure(mm Hg)"]
    + 2 * df["Diastolic Blood Pressure(mm Hg)"]
) / 3

# Convert Fahrenheit to Celsius
df["Temperature_C"] = (
    df["Body Temperature(F)"] - 32
) * 5 / 9

# Create age groups
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 18, 25, 35, 45, np.inf],
    labels=["Under 18", "18-24", "25-34", "35-44", "45+"],
    right=False
)

# Create BMI categories
df["BMI_Category"] = pd.cut(
    df["BMI(kg/m 2)"],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=["Underweight", "Normal", "Overweight", "Obese"],
    right=False
)

print("New features created successfully!")
display(df.head())


New features created successfully!


,Age,Body Temperature(F),Heart rate(bpm),Systolic Blood Pressure(mm Hg),Diastolic Blood Pressure(mm Hg),BMI(kg/m 2),Blood Glucose(HbA1c),Blood Glucose(Fasting hour-mg/dl),Status,Pulse_Pressure,Mean_Arterial_Pressure,Temperature_C,Age_Group,BMI_Category
0,20,97.5,91,161,100,24.9,41,5.8,high risk,61,120.333333,36.388889,18-24,Normal
1,45,97.7,99,99,94,22.1,36,5.7,high risk,5,95.666667,36.500000,45+,Normal
2,29,98.6,84,129,87,19.0,42,6.4,mid risk,42,101.000000,37.000000,25-34,Normal
3,26,99.5,135,161,101,23.7,46,4.5,high risk,60,121.000000,37.500000,25-34,Normal
4,38,102.5,51,106,91,18.8,38,4.3,high risk,15,96.000000,39.166667,35-44,Normal


In [2]:

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/maternal_health_engineered.csv",
    index=False
)

print("Engineered dataset saved!")
print("Dataset shape:", df.shape)
print("New features:")
print([
    "Pulse_Pressure",
    "Mean_Arterial_Pressure",
    "Temperature_C",
    "Age_Group",
    "BMI_Category"
])


Engineered dataset saved!
Dataset shape: (6103, 14)
New features:
['Pulse_Pressure', 'Mean_Arterial_Pressure', 'Temperature_C', 'Age_Group', 'BMI_Category']


In [3]:

# Remove records without a target label
df_model = df.dropna(subset=["Status"]).copy()

# Separate input features and target
X = df_model.drop(columns=["Status"])
y = df_model["Status"].astype(str).str.strip()

# Remove any remaining identifier columns, if present
X = X.drop(
    columns=["Patient ID", "Name"],
    errors="ignore"
)

print("Input feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())


Input feature shape: (6103, 13)
Target shape: (6103,)

Target distribution:
Status
high risk    2059
mid risk     2043
low risk     2001
Name: count, dtype: int64


In [4]:

from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))

print("\nTraining percentage:", round(len(X_train)/len(X)*100, 2))
print("Validation percentage:", round(len(X_val)/len(X)*100, 2))
print("Testing percentage:", round(len(X_test)/len(X)*100, 2))


Training records: 4272
Validation records: 915
Testing records: 916

Training percentage: 70.0
Validation percentage: 14.99
Testing percentage: 15.01


In [5]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify numerical and categorical input columns
num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

cat_cols = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation and testing data without fitting again
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Validation processed shape:", X_val_processed.shape)
print("Testing processed shape:", X_test_processed.shape)


Training processed shape: (4272, 19)
Validation processed shape: (915, 19)
Testing processed shape: (916, 19)


In [6]:

from scipy import sparse

def save_processed(data, filename):
    if sparse.issparse(data):
        data = data.toarray()
    np.save(filename, data)

save_processed(
    X_train_processed,
    "../data/processed/X_train.npy"
)

save_processed(
    X_val_processed,
    "../data/processed/X_validation.npy"
)

save_processed(
    X_test_processed,
    "../data/processed/X_test.npy"
)

# Save target labels separately
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_val.to_csv("../data/processed/y_validation.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("All processed splits saved successfully!")


All processed splits saved successfully!


In [7]:

import joblib
import os

os.makedirs("../data/processed", exist_ok=True)

# Save the fitted preprocessing pipeline
joblib.dump(
    preprocessor,
    "../data/processed/preprocessing_pipeline.pkl"
)

# Save the transformed feature names
feature_names = preprocessor.get_feature_names_out()

pd.DataFrame({
    "Feature_Name": feature_names
}).to_csv(
    "../data/processed/processed_feature_names.csv",
    index=False
)

print("Preprocessing pipeline saved successfully!")
print("Number of processed features:", len(feature_names))


Preprocessing pipeline saved successfully!
Number of processed features: 19


In [8]:

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Validation", "Testing"],
    "Records": [
        len(X_train),
        len(X_val),
        len(X_test)
    ],
    "Percentage": [
        len(X_train) / len(X) * 100,
        len(X_val) / len(X) * 100,
        len(X_test) / len(X) * 100
    ]
})

display(split_summary)

split_summary.to_csv(
    "../docs/dataset_split_summary.csv",
    index=False
)


,Dataset,Records,Percentage
0,Training,4272,69.998361
1,Validation,915,14.992627
2,Testing,916,15.009012
